# Experimentul 4: referința clasică, regresie logistică pe aceleași intrări

**Obligatoriu în Task 2:** un model clasic antrenat pe **exact aceleași intrări** ca circuitul, cel puțin o regresie logistică. Fără el, un rezultat cuantic nu poate fi interpretat (notebookul oficial, Partea 9.2; Bowles, Ahmed și Schuld, 2024).

## Ce facem
Pentru fiecare seed `[0, 1, 2, 3, 4]`:
1. același set de antrenare 1:1 (`undersample(X_train, y_train, seed)`) și același `AngleEncoder(4, seed)` ca VQC-urile din Exp. 2 și Exp. 3, deci aceleași 4 unghiuri în $[0, \pi]$;
2. **verificăm identitatea intrărilor**: ponderile salvate ale Exp. 2 și Exp. 3, aplicate pe aceste unghiuri, trebuie să reproducă exact AUPRC-ul lor salvat la p = 0;
3. `LogisticRegression(max_iter=1000)`, exact ca în notebookul oficial, antrenată pe setul 1:1;
4. scoruri pe validation (56.962 de tranzacții, 99 de fraude, rata reală ~0,17%), pragul max-F1 pe validation, metricile la acest prag.

## Statutul rezultatelor
Toate valorile sunt **rezultate de dezvoltare pe validation**: pragul este ales și metricile sunt calculate pe același set, deci precision / recall / F1 / MCC sunt optimiste, exact ca la VQC-uri. **Nu** sunt rezultate finale; acestea vin din test, o singură dată, la final. Testul nu este folosit aici (`RUN_TEST = False`).

## Zgomotul
Regresia logistică rulează pe un calculator clasic: zgomotul depolarizant al porților cuantice nu o afectează. Scorurile ei sunt aceleași la orice p, deci în comparație apare ca **același reper** pentru p = 0, 0.001, 0.01 și 0.05. Spearman ρ „cu zgomot vs fără zgomot” nu are sens pentru ea (ar fi 1 prin construcție) și nu este raportat.

## 1. Configurație și imports

`LogisticRegression` cu solverul implicit (`lbfgs`) este deterministă: nu folosește numere aleatoare, deci seed-ul intervine doar prin setul 1:1 și encoder. `set_seed` fixează oricum numpy și torch.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

RUN_TEST = False              # testul nu se folosește în acest experiment

SPLIT_SEED = 42
N_QUBITS = 4                  # numărul de componente PCA = numărul de qubiți ai VQC-urilor
ANSATZ_REPS = 3               # doar pentru verificarea identității intrărilor cu VQC-urile
FEATURE_MAP_REPS = 1          # setat temporar la 2 sau 1 în verificare (Exp. 2 / Exp. 3)
LR_MAX_ITER = 1000            # ca în notebookul oficial
NOISE_LEVELS = [0.001, 0.01, 0.05]
SEEDS = [0, 1, 2, 3, 4]
P_ALL = [0.0, *NOISE_LEVELS]

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp4"
EXP2_DIR = REPO_ROOT / "results" / "result_exp2"   # baseline VQC (ZZ ×2): doar citit
EXP3_DIR = REPO_ROOT / "results" / "result_exp3"   # VQC simplificat (ZZ ×1): doar citit
SPLIT_PATH = REPO_ROOT / "split_indices.npz"
VQC_REFERENCES = {"Exp. 2": (EXP2_DIR, 2), "Exp. 3": (EXP3_DIR, 1)}   # director, repetări ZZ

LR_LABEL, BASE_LABEL, NEW_LABEL = "regresie logistică (Exp. 4)", "VQC baseline Exp. 2 (ZZ ×2)", "VQC Exp. 3 (ZZ ×1)"
BASE_COLOR, NEW_COLOR, LR_COLOR, SEED_DOT_COLOR = "#2a78d6", "#eb6834", "#3a3a3a", "#8a8f98"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
print(f"seeds: {SEEDS}   RUN_TEST: {RUN_TEST}")
print("sklearn", sklearn.__version__, "| pennylane", qml.__version__, "| torch", torch.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele și splitul comun

Identic cu Experimentul 3: `split_indices.npz` (obligatoriu), verificat față de reconstrucția oficială. Features-urile testului nu rămân în memorie.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
for required in [DATA_PATH, SPLIT_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {required}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns and df.index.equals(pd.RangeIndex(len(df)))

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), "split_indices.npz diferă de splitul oficial"
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
assert (int(y_train.sum()), int(y_val.sum()), int(y_all[test_idx].sum())) == (295, 99, 98) and len(test_idx) == 56962
del X_all, df, _trval, _test, _train, _val   # features-urile testului nu rămân în memorie

print(f"split: {SPLIT_PATH.name} (identic cu reconstrucția oficială)")
print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())} fraude, rată {y_val.mean():.3%})")

## 3. Funcțiile din notebookul oficial

`undersample`, `AngleEncoder`, `choose_threshold`, `evaluate` sunt cele din notebookul oficial (identice cu Exp. 2 și Exp. 3). Circuitul VQC (feature map, ansatz, `predict`) este inclus **doar** pentru verificarea identității intrărilor; nu se antrenează nimic cuantic aici.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


# ---- VQC-ul, doar pentru verificarea identității intrărilor ----
def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


device = qml.device("default.qubit", wires=N_QUBITS)
circuit = qml.QNode(vqc, device, interface="torch", diff_method="backprop")


def fraud_probability(x, weights):
    return (1 - circuit(x, weights)) / 2


def predict(Z, weights, model=None, chunk=20000):
    model = model or fraud_probability
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()


def vqc_scores(Z, weights, reps):
    """Scorurile fără zgomot ale VQC-ului cu `reps` repetări ZZ (schimbare temporară, restaurată în finally)."""
    global FEATURE_MAP_REPS
    saved_reps, FEATURE_MAP_REPS = FEATURE_MAP_REPS, reps
    try:
        return predict(Z, weights)
    finally:
        FEATURE_MAP_REPS = saved_reps


def load_jsonl(path, required=False):
    """Înregistrările din path. Un fișier obligatoriu lipsă oprește execuția, în loc să întoarcă o listă goală."""
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


# ✅ Check 9.1 (din notebook): pragul maximizează F1
rng_check = np.random.default_rng(3)
y_c = rng_check.integers(0, 2, 300)
s_c = np.clip(0.3 * y_c + rng_check.random(300), 0, 1)
best_f1 = max(f1_score(y_c, (s_c >= t).astype(int)) for t in np.unique(s_c))
assert np.isclose(f1_score(y_c, (s_c >= choose_threshold(y_c, s_c)).astype(int)), best_f1)
print("✅ Check 9.1: trecut")

## 4. Intrări identice cu VQC-urile, apoi regresia logistică

Pentru fiecare seed construim setul 1:1 și unghiurile, apoi:
- **verificarea identității**: ponderile salvate ale Exp. 2 (ZZ ×2) și Exp. 3 (ZZ ×1) aplicate pe aceste unghiuri de validation trebuie să dea exact AUPRC-ul salvat de acele experimente la p = 0 (un forward fără zgomot, fără antrenare). Verificăm și că setul de antrenare are 590 de exemple, ca în `train_runs.jsonl` al Exp. 3;
- `LogisticRegression(max_iter=1000).fit(Z_train, y_train_1:1)`, scoruri `predict_proba` pe validation, pragul max-F1, metricile.

Rezultatele se salvează în `val_runs.jsonl` (o linie per seed, cu coeficienții modelului); un seed deja salvat nu se recalculează.

In [ ]:
config = {
    "experiment": "logistic_regression_reference", "model": f"sklearn LogisticRegression(max_iter={LR_MAX_ITER}), other parameters default",
    "inputs": "same as VQC: undersample(X_train, y_train, seed) 1:1 + AngleEncoder(4, seed) fitted on the 1:1 set, 4 angles in [0, pi]",
    "input_identity_check": "saved Exp. 2 and Exp. 3 weights reproduce their saved p = 0 validation AUPRC on these angles",
    "seeds": SEEDS, "threshold": "max F1 on validation (choose_threshold)", "evaluation_set": "validation (development results, test not used)",
    "noise": "none: classical model, identical scores at every p", "split": "split_indices.npz (verified against official reconstruction)",
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pennylane": qml.__version__, "torch": torch.__version__},
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH, VAL_RUNS_PATH = RESULTS_DIR / "config.json", RESULTS_DIR / "val_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: mută directorul sau schimbă RESULTS_DIR.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")

vqc_p0 = {name: {r["seed"]: r for r in load_jsonl(directory / "val_runs.jsonl", required=True) if r["p"] == 0.0}
          for name, (directory, _) in VQC_REFERENCES.items()}
exp3_train = {r["seed"]: r for r in load_jsonl(EXP3_DIR / "train_runs.jsonl", required=True)}
lr_done = {r["seed"] for r in load_jsonl(VAL_RUNS_PATH)}

identity_rows = []
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)
    Z_tr, Z_val = encoder.transform(X_s), encoder.transform(X_val)
    assert Z_tr.shape == (590, N_QUBITS) and int(y_s.sum()) == 295 and exp3_train[seed]["exemple train"] == len(y_s)
    assert np.allclose(Z_tr.min(axis=0), 0) and np.allclose(Z_tr.max(axis=0), np.pi) and Z_val.min() >= 0 and Z_val.max() <= np.pi

    identity = {"seed": seed}
    for name, (directory, reps) in VQC_REFERENCES.items():
        weights = torch.load(directory / "weights" / f"seed_{seed}.pt", weights_only=True)["weights"]
        ap = average_precision_score(y_val, vqc_scores(Z_val, weights, reps))
        saved_ap = vqc_p0[name][seed]["AUPRC"]
        assert np.isclose(ap, saved_ap, rtol=0, atol=1e-12), f"seed {seed}: intrările diferă de {name} (AUPRC {ap:.10f} vs {saved_ap:.10f})"
        identity[f"{name}: AUPRC recalculat"], identity[f"{name}: AUPRC salvat"] = ap, saved_ap
    identity_rows.append(identity)

    if seed in lr_done:
        print(f"seed {seed}: intrări identice cu Exp. 2 și Exp. 3; regresia logistică deja salvată, sărit")
        continue
    set_seed(seed)
    model = LogisticRegression(max_iter=LR_MAX_ITER).fit(Z_tr, y_s)
    assert model.n_iter_[0] < LR_MAX_ITER, f"seed {seed}: regresia logistică nu a convers"
    scores = model.predict_proba(Z_val)[:, 1]
    threshold = choose_threshold(y_val, scores)
    metrics = evaluate(y_val, scores, threshold)
    append_jsonl(VAL_RUNS_PATH, {"seed": seed, **metrics, "threshold": float(threshold), "iterații lbfgs": int(model.n_iter_[0]),
                                 "coeficienți": model.coef_[0], "intercept": float(model.intercept_[0])})
    print(f"seed {seed}: intrări identice cu Exp. 2 și Exp. 3   LR val AUPRC {metrics['AUPRC']:.3f}  F1 {metrics['F1']:.3f}  prag {threshold:.3f}  ({model.n_iter_[0]} iterații)")

identity = pd.DataFrame(identity_rows).set_index("seed")
identity.to_csv(RESULTS_DIR / "input_identity_check.csv")
print("✅ intrări identice: ponderile Exp. 2 și Exp. 3 reproduc exact AUPRC-ul lor salvat pe unghiurile folosite de regresia logistică")
display(identity.round(6))

## 5. Rezultate și comparația cu VQC-urile

**Rezultate de dezvoltare pe validation** (pragul ales pe același set). VQC-urile sunt citite din `results/result_exp2/val_runs.jsonl` și `results/result_exp3/val_runs.jsonl`. Regresia logistică are o singură valoare per seed, repetată identic la fiecare p. Diferențele sunt **pereche** (același seed = același set 1:1, același encoder), medie ± std (ddof = 1) pe 5 seeds.

In [ ]:
METRICS = ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC"]


def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


def load_vqc(directory):
    records = [r for r in load_jsonl(directory / "val_runs.jsonl", required=True) if r["seed"] in SEEDS and r["p"] in P_ALL]
    keys = [(r["seed"], r["p"]) for r in records]
    assert set(keys) == {(s, p) for s in SEEDS for p in P_ALL} and len(keys) == len(set(keys)), f"rezultate incomplete în {directory}"
    return pd.DataFrame(records)


lr = pd.DataFrame(load_jsonl(VAL_RUNS_PATH, required=True)).set_index("seed").loc[SEEDS]
assert lr.index.is_unique
vqc_results = {BASE_LABEL: load_vqc(EXP2_DIR), NEW_LABEL: load_vqc(EXP3_DIR)}

lr_cols = [*METRICS, "threshold", "TP", "FP", "FN", "TN", "iterații lbfgs", "intercept"]
print(f"Regresie logistică, per seed — validation (rezultate de dezvoltare, prag ales pe validation)")
display(lr[lr_cols].round(4))
lr[lr_cols].to_csv(RESULTS_DIR / "val_detailed.csv")
lr_summary = pd.DataFrame({m: [mean_std(lr[m])] for m in [*METRICS, "threshold"]}, index=[LR_LABEL])
print(f"Regresie logistică, medie ± std pe {len(SEEDS)} seeds — validation (aceeași la orice p)")
display(lr_summary)
lr_summary.to_csv(RESULTS_DIR / "val_summary.csv")

rows = []
for p in P_ALL:
    rows.append({"p": p, "model": LR_LABEL, **{m: mean_std(lr[m]) for m in METRICS}, "Spearman rho": "n/a (clasic)"})
    for label, results in vqc_results.items():
        part = results[results["p"] == p].set_index("seed").loc[SEEDS]
        rows.append({"p": p, "model": label, **{m: mean_std(part[m]) for m in METRICS}, "Spearman rho": mean_std(part["Spearman rho"], 4)})
comparison = pd.DataFrame(rows).set_index(["p", "model"])
comparison.to_csv(RESULTS_DIR / "comparison_lr_vs_vqc.csv")
print("Comparație pe validation, medie ± std pe 5 seeds (rezultate de dezvoltare)")
display(comparison)

paired_rows, per_seed_rows = [], []
for p in P_ALL:
    for label, results in vqc_results.items():
        part = results[results["p"] == p].set_index("seed").loc[SEEDS]
        for m in ["AUPRC", "AUROC", "F1", "MCC"]:
            diff = part[m] - lr[m]
            paired_rows.append({"p": p, "VQC": label, "metrică": m, "diferență pereche (VQC − LR)": f"{diff.mean():+.3f} ± {diff.std(ddof=1):.3f}",
                                "seeds cu VQC > LR": f"{int((diff > 0).sum())} / {len(diff)}"})
    for seed in SEEDS:
        row = {"p": p, "seed": seed, "AUPRC LR": lr.loc[seed, "AUPRC"]}
        for label, short in [(BASE_LABEL, "Exp. 2"), (NEW_LABEL, "Exp. 3")]:
            value = vqc_results[label].set_index(["seed", "p"]).loc[(seed, p), "AUPRC"]
            row[f"AUPRC {short}"], row[f"Δ AUPRC {short} − LR"] = value, value - lr.loc[seed, "AUPRC"]
        per_seed_rows.append(row)
paired = pd.DataFrame(paired_rows).set_index(["p", "VQC", "metrică"])
per_seed = pd.DataFrame(per_seed_rows)
paired.to_csv(RESULTS_DIR / "paired_differences_vs_lr.csv")
per_seed.to_csv(RESULTS_DIR / "paired_auprc_per_seed.csv", index=False)
print("Diferențe pereche VQC − regresie logistică (același seed)")
display(paired.loc[[0.0, 0.01, 0.05]])
print("AUPRC per seed la p = 0, 0.01, 0.05")
display(per_seed[per_seed["p"].isin([0.0, 0.01, 0.05])].set_index(["p", "seed"]).round(4))

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels = ["0", *[str(p) for p in NOISE_LEVELS]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col in zip(axes, ["AUPRC", "F1", "MCC"]):
    m, s = lr[col].mean(), lr[col].std(ddof=1)
    ax.axhspan(m - s, m + s, color=LR_COLOR, alpha=0.10, zorder=1)
    ax.axhline(m, color=LR_COLOR, linestyle="--", linewidth=2, label=f"{LR_LABEL}, ± std", zorder=2)
    for (label, results), color, marker, dx in zip(vqc_results.items(), [BASE_COLOR, NEW_COLOR], ["o", "s"], [-0.07, 0.07]):
        groups = [results.loc[results["p"] == p, col].to_numpy(dtype=float) for p in P_ALL]
        for i, g in enumerate(groups):
            ax.scatter(np.full(len(g), xpos[i] + dx), g, s=18, color=SEED_DOT_COLOR, alpha=0.6, zorder=2)
        ax.errorbar(xpos + dx, [g.mean() for g in groups], yerr=[g.std(ddof=1) for g in groups], color=color, marker=marker,
                    markersize=8, linewidth=2, capsize=4, label=label, zorder=3)
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă (doar VQC)")
    ax.set_title(f"{col} vs p" + (" (prag max-F1 la același p)" if col != "AUPRC" else ""))
axes[0].legend(loc="lower left", fontsize=8)
fig.suptitle("Validation (rezultate de dezvoltare) · regresie logistică vs VQC pe aceleași 4 unghiuri · medie ± std pe 5 seeds (gri: seeds)", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "comparison_lr_vs_vqc.png", dpi=130)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (label, short, color) in zip(axes, [(BASE_LABEL, "Exp. 2", BASE_COLOR), (NEW_LABEL, "Exp. 3", NEW_COLOR)]):
    for seed in SEEDS:
        d = per_seed[per_seed["seed"] == seed].set_index("p").loc[P_ALL, f"Δ AUPRC {short} − LR"]
        ax.plot(xpos, d, color=SEED_DOT_COLOR, marker="o", markersize=5, linewidth=1, alpha=0.8, zorder=2)
        ax.annotate(f"s{seed}", (xpos[-1], d.iloc[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
    ax.plot(xpos, per_seed.groupby("p")[f"Δ AUPRC {short} − LR"].mean().loc[P_ALL], color=color, marker="s", markersize=8, linewidth=2, label="medie pe 5 seeds", zorder=3)
    ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_title(f"Δ AUPRC: {label} − regresie logistică")
    ax.legend(loc="lower left")
axes[0].set_ylabel("Δ AUPRC (< 0: regresia logistică mai bună)")
fig.suptitle("Diferențe pereche pe seed (gri: câte o linie per seed) · validation", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "paired_auprc_vs_lr.png", dpi=130)
plt.show()

## 6. Cost și citire

**Cost cuantic al regresiei logistice: zero execuții, zero shots.** Se antrenează în câteva milisecunde pe CPU. VQC-urile costă ~3,09 milioane de execuții per seed (antrenare + validare, vezi ledger-ele Exp. 2 și Exp. 3).

**Cum citim comparația:**
- Regresia logistică vede aceleași 4 unghiuri ca VQC-ul; întrebarea este dacă circuitul extrage din ele mai mult decât un model liniar simplu.
- Regresia logistică nu este afectată de zgomot cuantic: linia ei este aceeași la toate nivelurile p. VQC-urile sunt comparate cu ea la fiecare p.
- Toate valorile sunt rezultate de dezvoltare pe validation (pragul ales pe același set; la VQC și epoca este aleasă pe validation). Validation are 99 de fraude.
- Rezultatele finale vin din test, o singură dată, pentru configurația aleasă.